# Exploratory Data Analysis — Credit Risk Model
**Bati Bank × Xente eCommerce | KAIM Week 4**

**Objective:** Explore the Xente transaction dataset to uncover patterns, identify data quality issues, understand feature distributions, and generate hypotheses to guide feature engineering and proxy target variable construction.

**Dataset:** Xente eCommerce transaction records (sourced from Kaggle Xente Challenge)

---

## 0. Setup — Imports and Configuration

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
DATA_PATH = '../data/raw/xente_data.csv'   # adjust path if needed

print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## 1. Dataset Overview

**Objective:** Understand the structure of the dataset — number of rows, columns, data types, and a first look at the raw data.

**Analysis steps:**
1. Load the dataset
2. Inspect shape, column names, and dtypes
3. Display sample rows
4. Count unique values per column

**Interpretation guidance:** Look for any immediately obvious issues — columns that should be numeric but are stored as object, date columns not parsed as datetime, or columns with suspiciously low cardinality (potential flags/binary features).

In [43]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Column names:')
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/xente_data.csv'

In [ ]:
print('Data types and non-null counts:')
print(df.info())

In [ ]:
print('First 5 rows:')
df.head()

In [ ]:
# Cardinality of each column
cardinality = df.nunique().reset_index()
cardinality.columns = ['Column', 'Unique Values']
cardinality['Dtype'] = df.dtypes.values
cardinality['% Unique'] = (cardinality['Unique Values'] / len(df) * 100).round(2)
print(cardinality.to_string(index=False))

In [ ]:
# Parse TransactionStartTime as datetime
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

print('Transaction date range:')
print(f"  Earliest: {df['TransactionStartTime'].min()}")
print(f"  Latest:   {df['TransactionStartTime'].max()}")
print(f"  Span:     {(df['TransactionStartTime'].max() - df['TransactionStartTime'].min()).days} days")

---
## 2. Summary Statistics

**Objective:** Understand the central tendency, dispersion, and shape of numerical feature distributions.

**Analysis steps:**
1. Compute descriptive statistics for numerical features
2. Identify highly skewed features (|skewness| > 1)
3. Flag columns with extreme kurtosis (potential outlier-heavy distributions)

**Interpretation guidance:** High skewness in `Amount` and `Value` is expected for transaction data — a small number of high-value transactions dominate. Kurtosis >> 3 suggests heavy tails and potential outliers requiring treatment.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numerical columns: {numerical_cols}')
print()
df[numerical_cols].describe().T

In [ ]:
# Skewness and kurtosis
skew_kurt = pd.DataFrame({
    'Skewness': df[numerical_cols].skew(),
    'Kurtosis': df[numerical_cols].kurtosis()
})
skew_kurt['Highly Skewed'] = skew_kurt['Skewness'].abs() > 1
skew_kurt['Heavy Tails']   = skew_kurt['Kurtosis'].abs() > 3
print(skew_kurt.round(3).to_string())

In [ ]:
# FraudResult value counts (binary target)
fraud_counts = df['FraudResult'].value_counts()
fraud_pct = df['FraudResult'].value_counts(normalize=True) * 100
print('FraudResult distribution:')
for label, count in fraud_counts.items():
    print(f"  {label}: {count:,} ({fraud_pct[label]:.2f}%)")

---
## 3. Distribution of Numerical Features

**Objective:** Visualize numerical feature distributions to identify patterns, skewness, bimodality, and potential data entry errors.

**Analysis steps:**
1. Plot histograms with KDE for `Amount` and `Value`
2. Compare raw vs. log-transformed distributions
3. Plot transaction volume over time

**Interpretation guidance:** `Amount` may include both positive (debits) and negative (credits/refunds) values — this bimodal distribution is expected. Log transformation helps visualize the spread of the positive tail. Temporal plots reveal seasonality or data collection gaps.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount — raw
axes[0, 0].hist(df['Amount'], bins=100, edgecolor='none', color='steelblue', alpha=0.75)
axes[0, 0].set_title('Amount Distribution (Raw)', fontsize=13)
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['Amount'].median(), color='red', linestyle='--', label=f"Median: {df['Amount'].median():,.0f}")
axes[0, 0].legend()

# Amount — positive only, log scale
pos_amount = df[df['Amount'] > 0]['Amount']
axes[0, 1].hist(np.log1p(pos_amount), bins=60, edgecolor='none', color='steelblue', alpha=0.75)
axes[0, 1].set_title('Amount Distribution — Positive Values (log1p)', fontsize=13)
axes[0, 1].set_xlabel('log1p(Amount)')
axes[0, 1].set_ylabel('Frequency')

# Value — raw
axes[1, 0].hist(df['Value'], bins=100, edgecolor='none', color='coral', alpha=0.75)
axes[1, 0].set_title('Value Distribution (Raw)', fontsize=13)
axes[1, 0].set_xlabel('Value')
axes[1, 0].set_ylabel('Frequency')

# Value — log
axes[1, 1].hist(np.log1p(df['Value']), bins=60, edgecolor='none', color='coral', alpha=0.75)
axes[1, 1].set_title('Value Distribution (log1p)', fontsize=13)
axes[1, 1].set_xlabel('log1p(Value)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Numerical Feature Distributions', fontsize=15, y=1.02)
plt.savefig('../notebooks/figures/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os
os.makedirs('../notebooks/figures', exist_ok=True)

# Transaction volume over time
df_time = df.set_index('TransactionStartTime').resample('W')['TransactionId'].count()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(df_time.index, df_time.values, alpha=0.4, color='steelblue')
ax.plot(df_time.index, df_time.values, color='steelblue', linewidth=1.5)
ax.set_title('Weekly Transaction Volume Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Transactions')
plt.tight_layout()
plt.savefig('../notebooks/figures/transaction_volume_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak week: {df_time.idxmax().date()} with {df_time.max():,} transactions')

In [ ]:
# Amount: positive vs negative split
positive_pct = (df['Amount'] > 0).mean() * 100
negative_pct = (df['Amount'] < 0).mean() * 100
zero_pct     = (df['Amount'] == 0).mean() * 100

print(f'Amount breakdown:')
print(f'  Positive (debits):  {positive_pct:.1f}%')
print(f'  Negative (credits): {negative_pct:.1f}%')
print(f'  Zero:               {zero_pct:.1f}%')

---
## 4. Distribution of Categorical Features

**Objective:** Understand the frequency and variability of categorical features. Identify dominant categories, rare categories that may need grouping, and imbalances that could affect encoding strategies.

**Analysis steps:**
1. Plot bar charts for `ProductCategory`, `ChannelId`, `PricingStrategy`, `CurrencyCode`
2. Compute fraud rate by category to identify risk-associated categories

**Interpretation guidance:** Highly imbalanced categories (one category >80% of records) may warrant target encoding or grouping of rare categories. Fraud rate variation across `ProductCategory` is a key signal for feature engineering.

In [ ]:
categorical_cols = ['ProductCategory', 'ChannelId', 'PricingStrategy', 'CurrencyCode', 'CountryCode']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    vc = df[col].value_counts()
    axes[i].barh(vc.index.astype(str)[:15], vc.values[:15], color='steelblue', alpha=0.8)
    axes[i].set_title(f'{col} Distribution', fontsize=12)
    axes[i].set_xlabel('Count')
    for spine in ['top', 'right']:
        axes[i].spines[spine].set_visible(False)

# Hide the 6th (unused) subplot
axes[5].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=15)
plt.tight_layout()
plt.savefig('../notebooks/figures/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fraud rate by ProductCategory
fraud_by_cat = df.groupby('ProductCategory')['FraudResult'].agg(['mean', 'sum', 'count'])
fraud_by_cat.columns = ['FraudRate', 'FraudCount', 'TotalTransactions']
fraud_by_cat = fraud_by_cat.sort_values('FraudRate', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(fraud_by_cat.index, fraud_by_cat['FraudRate'] * 100, color='salmon', alpha=0.85)
ax.set_xlabel('Fraud Rate (%)')
ax.set_title('Fraud Rate by Product Category', fontsize=14)
for bar, count in zip(bars, fraud_by_cat['FraudCount']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'n={count}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../notebooks/figures/fraud_rate_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print(fraud_by_cat)

In [ ]:
# Fraud rate by ChannelId
fraud_by_channel = df.groupby('ChannelId')['FraudResult'].agg(['mean', 'sum', 'count'])
fraud_by_channel.columns = ['FraudRate', 'FraudCount', 'TotalTransactions']
fraud_by_channel = fraud_by_channel.sort_values('FraudRate', ascending=False)
print('Fraud rate by channel:')
print(fraud_by_channel.round(4))

---
## 5. Correlation Analysis

**Objective:** Understand linear relationships between numerical features. Identify multicollinear features (|r| > 0.8) that may cause issues in Logistic Regression and assess which features correlate most strongly with `FraudResult`.

**Analysis steps:**
1. Compute Pearson correlation matrix for numerical features
2. Plot heatmap
3. Identify and report top features correlated with `FraudResult`

**Interpretation guidance:** `Amount` and `Value` are expected to be highly correlated (Value is the absolute of Amount). High correlation between these two confirms that one may be redundant in the feature set. Any feature with near-zero correlation with `FraudResult` at the univariate level may still contribute in a multivariate model.

In [ ]:
corr_matrix = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix — Numerical Features', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature correlation with FraudResult
fraud_corr = df[numerical_cols].corrwith(df['FraudResult']).sort_values(key=abs, ascending=False)
print('Feature correlations with FraudResult:')
print(fraud_corr.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['salmon' if c < 0 else 'steelblue' for c in fraud_corr]
ax.barh(fraud_corr.index, fraud_corr.values, color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with FraudResult', fontsize=13)
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../notebooks/figures/fraud_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Missing Value Analysis

**Objective:** Identify the extent, pattern, and likely cause of missing values to determine appropriate imputation or removal strategies.

**Analysis steps:**
1. Compute null count and percentage per column
2. Visualize missingness heatmap
3. Assess whether missingness is random (MAR) or informative (MNAR)

**Interpretation guidance:** If missingness clusters in specific rows (e.g., all nulls are for a particular `ProductCategory`), it may be MNAR and should be modeled explicitly (e.g., missingness indicator feature). Random nulls across rows suggest MAR and are amenable to median/mode imputation.

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing.empty:
    print('✅ No missing values detected in the dataset.')
else:
    print(f'Columns with missing values ({len(missing)} columns):')
    print(missing.to_string())
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.5)))
    ax.barh(missing.index, missing['Missing %'], color='salmon', alpha=0.8)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Value Rate by Column', fontsize=14)
    for i, (idx, row) in enumerate(missing.iterrows()):
        ax.text(row['Missing %'] + 0.1, i, f"{row['Missing %']:.1f}%", va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('../notebooks/figures/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Check for duplicate TransactionIds
dup_count = df.duplicated(subset='TransactionId').sum()
print(f'Duplicate TransactionIds: {dup_count}')

# Check for duplicate rows entirely
full_dup = df.duplicated().sum()
print(f'Fully duplicate rows: {full_dup}')

# Unique customers
print(f"Unique Customers (CustomerId): {df['CustomerId'].nunique():,}")
print(f"Unique Accounts (AccountId):   {df['AccountId'].nunique():,}")
print(f"Total Transactions:            {len(df):,}")
print(f"Avg transactions/customer:     {len(df) / df['CustomerId'].nunique():.1f}")

---
## 7. Outlier Detection

**Objective:** Identify extreme values in numerical features using box plots and IQR-based rules. Determine whether outliers represent data quality issues or legitimate high-value transactions.

**Analysis steps:**
1. Plot box plots for `Amount` and `Value`
2. Apply IQR rule (Q1 - 1.5×IQR, Q3 + 1.5×IQR) to quantify outliers
3. Assess whether outliers are disproportionately associated with fraud

**Interpretation guidance:** In financial transaction data, extreme values often represent genuine high-value purchases, not errors. Removing them would distort RFM features. Capping (Winsorization) at the 1st/99th percentile is typically preferred over deletion. Check whether outlier transactions overlap with fraudulent transactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot — Amount
axes[0].boxplot(df['Amount'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Box Plot — Amount', fontsize=13)
axes[0].set_ylabel('Amount')

# Box plot — Value
axes[1].boxplot(df['Value'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='coral', alpha=0.5),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Box Plot — Value', fontsize=13)
axes[1].set_ylabel('Value')

plt.suptitle('Outlier Detection — Numerical Features', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/figures/outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def iqr_outlier_summary(series, name):
    """Compute IQR-based outlier statistics for a numeric series."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((series < lower) | (series > upper)).sum()
    pct = n_outliers / len(series) * 100
    print(f'{name}:')
    print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
    print(f'  Lower fence: {lower:.2f} | Upper fence: {upper:.2f}')
    print(f'  Outliers: {n_outliers:,} ({pct:.2f}%)')
    print()
    return lower, upper

lower_amount, upper_amount = iqr_outlier_summary(df['Amount'], 'Amount')
lower_value,  upper_value  = iqr_outlier_summary(df['Value'],  'Value')

In [ ]:
# Fraud rate among outlier vs non-outlier transactions
df['amount_outlier'] = (df['Amount'] < lower_amount) | (df['Amount'] > upper_amount)

fraud_outlier = df.groupby('amount_outlier')['FraudResult'].mean() * 100
print('Fraud rate — Amount outliers vs non-outliers:')
for k, v in fraud_outlier.items():
    label = 'Outlier' if k else 'Non-Outlier'
    print(f'  {label}: {v:.2f}%')

# Clean up temp column
df.drop(columns=['amount_outlier'], inplace=True)

In [ ]:
# Per-customer transaction count distribution (input to RFM Frequency)
txn_per_customer = df.groupby('CustomerId')['TransactionId'].count()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(txn_per_customer, bins=50, edgecolor='none', color='mediumpurple', alpha=0.8)
ax.set_title('Distribution of Transactions per Customer', fontsize=13)
ax.set_xlabel('Number of Transactions')
ax.set_ylabel('Number of Customers')
ax.axvline(txn_per_customer.median(), color='red', linestyle='--',
           label=f'Median: {txn_per_customer.median():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('../notebooks/figures/txn_per_customer.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Customers with only 1 transaction: {(txn_per_customer == 1).sum():,} "
      f"({(txn_per_customer == 1).mean()*100:.1f}% of customers)")
print(f"Max transactions by a single customer: {txn_per_customer.max():,}")

---
## 8. RFM Feature Preview

**Objective:** Compute a preliminary RFM (Recency, Frequency, Monetary) profile per customer as a preview for Task 4 proxy target engineering. Visualize the RFM distributions to understand the range of customer engagement levels.

**Analysis steps:**
1. Define snapshot date as the day after the last transaction in the dataset
2. Compute Recency (days since last transaction), Frequency (transaction count), Monetary (total transaction amount) per customer
3. Visualize distributions and pairwise relationships

In [ ]:
snapshot_date = df['TransactionStartTime'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerId').agg(
    Recency   = ('TransactionStartTime', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('TransactionId', 'count'),
    Monetary  = ('Amount', lambda x: x[x > 0].sum())  # sum of positive (debit) amounts
).reset_index()

print(f'RFM table shape: {rfm.shape}')
print(rfm.describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rfm_cols = ['Recency', 'Frequency', 'Monetary']
colors = ['steelblue', 'mediumpurple', 'coral']

for ax, col, color in zip(axes, rfm_cols, colors):
    ax.hist(rfm[col], bins=50, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(f'RFM — {col}', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Customers')
    ax.axvline(rfm[col].median(), color='red', linestyle='--',
               label=f'Median: {rfm[col].median():.0f}')
    ax.legend(fontsize=9)

plt.suptitle('RFM Distribution Across Customers', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/figures/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pairplot of RFM (sample for speed)
rfm_sample = rfm[rfm_cols].sample(min(2000, len(rfm)), random_state=RANDOM_STATE)

g = sns.pairplot(rfm_sample, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 15},
                 diag_kws={'color': 'steelblue'})
g.fig.suptitle('RFM Pairplot', y=1.02, fontsize=14)
plt.savefig('../notebooks/figures/rfm_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 📌 Top EDA Insights

The following five insights summarize the most important findings from this exploratory analysis. They directly inform the feature engineering and proxy target construction in subsequent tasks.

---

### Insight 1: Extreme Class Imbalance in FraudResult

The `FraudResult` column (which serves as a behavioral signal and potential feature) is **highly imbalanced**, with fewer than 1% of transactions flagged as fraudulent. This imbalance means that naive accuracy metrics will be misleading. Any model trained on this dataset must use **stratified sampling**, **SMOTE**, or **class-weight adjustments**, and should be evaluated using **Precision-Recall AUC** and **F1-Score** rather than overall accuracy.

---

### Insight 2: `Amount` and `Value` Are Nearly Perfectly Correlated

`Value` is the absolute value of `Amount`. The Pearson correlation between these two features is near 1.0 for positive transactions and the pair carries **redundant information**. Only one of these should be retained in the feature set to avoid multicollinearity — particularly important for Logistic Regression with WoE. The `Amount` feature additionally encodes the direction of the transaction (positive = debit, negative = credit/refund), which carries distinct information about customer behavior and should be engineered separately.

---

### Insight 3: Highly Skewed Transaction Amounts Requiring Log Transformation

The distribution of `Amount` (and `Value`) is **extremely right-skewed**, with the majority of transactions being small and a long tail of high-value transactions. Without log transformation or binning, these features will violate the normality assumptions of Logistic Regression and may cause gradient-based models to overweight extreme values. **log1p transformation** is the recommended preprocessing step, and WoE binning will naturally handle skewness by converting the continuous variable to discrete risk categories.

---

### Insight 4: Customer Transaction Frequency Is Highly Heterogeneous — Critical for RFM Segmentation

A substantial proportion of customers in the dataset have **only 1–2 transactions**, while a small group of highly active customers transact dozens of times. This bimodal customer activity distribution means that the **Frequency** component of RFM will be a strong discriminator in K-Means clustering. Customers with single transactions are disproportionately likely to be classified as high-risk under the proxy model, since their behavioral signal is minimal — this is both an expected modeling outcome and a risk that should be disclosed (single-purchase customers are not necessarily high credit risk).

---

### Insight 5: Product Category and Channel Are Strongly Predictive of Fraud Risk

Fraud rates vary substantially across `ProductCategory` and `ChannelId`. Certain categories show disproportionately high fraud rates, suggesting that **product category** and **transaction channel** are strong candidates for inclusion in the feature set. When these categorical variables are WoE-encoded, they will likely carry high Information Value (IV > 0.3 threshold for predictive strength). This also implies that the proxy risk model should capture behavioral patterns at the category level — not just aggregate RFM scores — for maximum discriminatory power.